# Real Photos, More Complexity: CIFAR-10

MNIST digits are grayscale, centered, and clean. **CIFAR-10** is the next step up: 60,000 real, small, color photographs across 10 categories (airplane, car, bird, cat, deer, dog, frog, horse, ship, truck).

What's new here:
- **Color images** (3 channels - red, green, blue - instead of 1).
- A **third convolutional block**, since color photos have more to learn than digits.
- **Batch normalization** and **dropout** - two techniques that help deeper networks train well and avoid memorizing the training data too closely.
- Honestly lower accuracy than MNIST - and that's the actual lesson here, not a bug.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

CLASSES = ["airplane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]


In [ ]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),   # a cheap way to make the training set effectively bigger
    transforms.ToTensor(),
])
test_transform = transforms.ToTensor()   # no augmentation at test time - we want to evaluate on the real images

train_data = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

print("Training examples:", len(train_data))
image, label = train_data[0]
print("One image shape:", image.shape, " (channels, height, width)  label:", CLASSES[label])


## The network

> 🏷️ **B1-K1-W3** · Realiseert (onderdelen van) software — kwalificatiedossier Software development, kerntaak B1-K1

Three conv blocks instead of two, each followed by batch normalization (`BatchNorm2d`) and, after flattening, `Dropout` before the final decision. Trace the sizes: 32x32 -> pool -> 16x16 -> pool -> 8x8 -> pool -> 4x4, so the flattened size is `64 * 4 * 4 = 1024`.

In [ ]:
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CifarCNN().to(device)
print(model)
print("\nTotal parameters:", sum(p.numel() for p in model.parameters()))


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

EPOCHS = 8

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = loss_fn(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    avg_loss = running_loss / len(train_data)

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
    accuracy = correct / len(test_data)

    print(f"epoch {epoch+1}/{EPOCHS}  |  loss {avg_loss:.4f}  |  test accuracy {accuracy:.1%}")


### Why is this so much harder than MNIST?

> 🏷️ **B1-K1-W4** · Test software — kwalificatiedossier Software development, kerntaak B1-K1

You likely landed somewhere around 70-78% accuracy - genuinely good for a network this size, but nowhere near MNIST's 97%+. That gap *is* the lesson: real photos have lighting, angle, background clutter, and genuine visual similarity between classes (a cat and a dog silhouette can look alike; a "2" and a "7" rarely do). This is exactly why real-world computer vision systems use bigger networks, more data, and - as the next notebook shows - a head start from a network someone else already trained.

### Kwalificatie-koppeling
Deze notebook dekt B1-K1-W3 (Realiseert (onderdelen van) software) en B1-K1-W4 (Test software) uit kerntaak B1-K1 van het kwalificatiedossier mbo Software development (Crebo 23399, gewijzigd 2024). Volledige dekking over alle lessen heen: https://projectenplaats.nl/kwalificatiedossiers/software-development